# DBSCAN From Scratch

Wiki reference for [the DBSCAN algorithm](https://ml-viz-ruby.vercel.app/wiki/dbscan-algorithm).

**The idea in one sentence.** DBSCAN clusters by **density**: a point is a **core** point if it
has at least `min_samples` neighbours within `eps`, clusters grow by chaining core points, and
anything not reachable is **noise** — so it finds arbitrarily-shaped clusters and labels
outliers, but is very sensitive to the `eps` you choose.

We implement region queries and the DBSCAN loop from scratch, **validate it against sklearn and
the core/noise labelling**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use('dark_background')
COLORS = {'core': '#6366f1', 'border': '#22d3ee', 'noise': '#f59e0b',
          'cluster0': '#6366f1', 'cluster1': '#22d3ee', 'noise_pt': '#f59e0b'}

## From-scratch DBSCAN

In [ ]:
def region_query(X, point_idx, eps):
    """Return indices of all points within eps of X[point_idx] (including itself)."""
    dists = np.linalg.norm(X - X[point_idx], axis=1)
    return np.where(dists <= eps)[0].tolist()

def dbscan(X, eps, min_samples):
    """
    Returns:
        labels: (N,) array — cluster id (0-indexed) or -1 for noise
        core_mask: (N,) bool array — True for core points
    """
    n = len(X)
    labels = np.full(n, -2, dtype=int)   # -2 = unvisited
    cluster_id = 0

    for i in range(n):
        if labels[i] != -2:              # already visited
            continue
        neighbors = region_query(X, i, eps)
        if len(neighbors) < min_samples:
            labels[i] = -1              # tentative noise
            continue
        # Start a new cluster
        labels[i] = cluster_id
        seed_set = list(neighbors)
        seed_set.remove(i)
        j = 0
        while j < len(seed_set):
            q = seed_set[j]
            if labels[q] == -1:
                labels[q] = cluster_id  # was noise, now border
            if labels[q] != -2:
                j += 1
                continue
            labels[q] = cluster_id
            q_neighbors = region_query(X, q, eps)
            if len(q_neighbors) >= min_samples:
                seed_set += [n for n in q_neighbors if n not in seed_set]
            j += 1
        cluster_id += 1

    # Compute core mask
    core_mask = np.array([
        len(region_query(X, i, eps)) >= min_samples for i in range(n)
    ])
    return labels, core_mask

## Worked example: 9 points, ε = 2, min_samples = 3

In [ ]:
# Blob 1: A-D, Blob 2: E-H, Isolated: I
X = np.array([[1,1],[1,2],[2,1],[2,2],   # A B C D
              [8,8],[8,9],[9,8],[9,9],   # E F G H
              [5,5]])                    # I
names = list('ABCDEFGHI')

labels, core_mask = dbscan(X, eps=2, min_samples=3)

print("Point  Cluster  Core?")
print("-----  -------  -----")
for name, lbl, is_core in zip(names, labels, core_mask):
    cluster = f"Cluster {lbl}" if lbl >= 0 else "Noise"
    print(f"  {name}      {cluster:<10}  {'yes' if is_core else 'no'}")

In [ ]:
# Verify against sklearn
from sklearn.cluster import DBSCAN
sk_labels = DBSCAN(eps=2, min_samples=3).fit_predict(X)
print(f"\nScratch labels: {labels}")
print(f"sklearn labels: {sk_labels}")
print(f"Match: {np.all(labels == sk_labels)}")

### Validate: the from-scratch DBSCAN matches sklearn

Our implementation should reproduce sklearn's labels exactly on the worked example, correctly
marking the isolated point as **noise** ($-1$) and a point inside a dense blob as a **core**
point. We confirm all three.

In [ ]:
from sklearn.cluster import DBSCAN as _SK
sk = _SK(eps=2, min_samples=3).fit_predict(X)
print(f'scratch: {labels}')
print(f'sklearn: {sk}')
assert np.all(labels == sk), 'the from-scratch DBSCAN matches sklearn label-for-label'
assert labels[8] == -1, 'the isolated point (I) is labelled noise'
assert core_mask[0], 'a point inside a dense blob (A) is a core point'
print('\n✅ density-reachability recovers the clusters and flags the outlier as noise')

## Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

palette = ['#6366f1', '#22d3ee', '#f59e0b']  # cluster0, cluster1, noise
markers = {True: 'o', False: 's'}  # circle=core, square=border/noise

for ax, (title, lbl_arr, cm) in zip(axes, [
    ('Point types (shape = role)', labels, core_mask),
    ('Cluster assignments', labels, core_mask)
]):
    for i, (pt, lbl, is_core) in enumerate(zip(X, lbl_arr, cm)):
        color = palette[lbl] if lbl >= 0 else palette[2]
        marker = 'o' if is_core else ('s' if lbl >= 0 else 'X')
        ax.scatter(*pt, c=color, s=200, marker=marker, zorder=3,
                   edgecolors='white', linewidths=0.8)
        ax.annotate(names[i], pt, textcoords='offset points',
                    xytext=(6, 4), color='white', fontsize=11)

    # Draw eps circles for core points
    for i, (pt, is_core) in enumerate(zip(X, cm)):
        if is_core:
            circle = plt.Circle(pt, 2, color=palette[lbl_arr[i]],
                                fill=False, alpha=0.25, linestyle='--')
            ax.add_patch(circle)

    ax.set_xlim(-1, 12); ax.set_ylim(-1, 12)
    ax.set_title(title, pad=10)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.15)

# Legend
legend_els = [
    mpatches.Patch(color='#6366f1', label='Cluster 0'),
    mpatches.Patch(color='#22d3ee', label='Cluster 1'),
    mpatches.Patch(color='#f59e0b', label='Noise'),
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='gray',
               markersize=10, label='Core point', linestyle='None'),
    plt.Line2D([0],[0], marker='X', color='w', markerfacecolor='gray',
               markersize=10, label='Noise point', linestyle='None'),
]
axes[1].legend(handles=legend_els, loc='upper left', framealpha=0.3)
plt.tight_layout()
plt.suptitle('DBSCAN: 9-point worked example (ε=2, min_samples=3)',
             y=1.02, fontsize=13)
plt.show()

## k-Distance elbow for ε selection

In [ ]:
from sklearn.neighbors import NearestNeighbors

k = 2  # min_samples - 1
nn = NearestNeighbors(n_neighbors=k + 1)  # +1 to exclude self
nn.fit(X)
distances, _ = nn.kneighbors(X)
k_dists = np.sort(distances[:, -1])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_dists, 'o-', color='#6366f1', markersize=8)
ax.axhline(2.0, color='#f59e0b', ls='--', lw=1.5, label='ε = 2 (chosen)')
ax.set_xlabel('Points (sorted by k-NN distance)')
ax.set_ylabel(f'{k}-NN distance')
ax.set_title('k-Distance plot — choose ε at the elbow')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **eps sensitivity** | too small = all noise, too large = one cluster (demo) |
| **varying density** | a single eps can't fit clusters of different densities (see HDBSCAN) |
| **high dimensions** | distances concentrate; density becomes uninformative |
| **min_samples** | controls what counts as dense; tune with eps together |
| **border points** | can belong to different clusters depending on processing order |

Demo: eps too small yields all-noise; too large yields one cluster.

In [ ]:
# DBSCAN's defining weakness is EPS sensitivity. Too small an eps and no point has enough
# neighbours -> everything is noise. Too large and every point is density-reachable from every
# other -> one giant cluster. The 'right' eps sits at the elbow of the k-distance plot; miss it
# and the clustering is meaningless. We show both failure extremes.
lab_small, _ = dbscan(X, eps=0.5, min_samples=3)
lab_large, _ = dbscan(X, eps=10, min_samples=3)
print(f'eps=0.5 -> {lab_small}   (all noise)')
print(f'eps=10  -> {lab_large}   (one cluster)')
assert (lab_small == -1).all(), 'eps too small: every point is noise'
assert len(set(lab_large[lab_large >= 0])) == 1, 'eps too large: everything merges into one cluster'
print('\nDBSCAN lives or dies by eps -> choose it at the k-distance elbow, not by guessing.')

## ✏️ Your turn

**Exercise 1:** Add three new points forming a crescent around Cluster 0 (e.g. `(0,3), (1.5,3.5), (3,3)`) and re-run `dbscan`. Do they join Cluster 0 or become noise? Adjust `eps` to make them join.

**Exercise 2:** Run DBSCAN on the `make_moons` dataset from sklearn with `eps=0.3` and `min_samples=5`. How many clusters do you find? Compare to K-Means (k=2) on the same data.

**Exercise 3:** Implement a `point_type` function that classifies each point as `'core'`, `'border'`, or `'noise'` using only the `region_query` helper and the DBSCAN labels.

In [ ]:
# Exercise 1
new_points = np.array([
    # TODO(you): add crescent points here
])
X_new = np.vstack([X, new_points])
labels_new, core_new = dbscan(X_new, eps=2, min_samples=3)
# TODO(you): print or plot results

In [ ]:
# Exercise 3
def point_type(X, labels, idx, eps, min_samples):
    """
    Returns 'core', 'border', or 'noise' for X[idx].
    """
    # TODO(you): use region_query and labels
    pass

# assert point_type(X, labels, 0, 2, 3) == 'core'
# assert point_type(X, labels, 8, 2, 3) == 'noise'

<details>
<summary>Solution — Exercise 3</summary>

```python
def point_type(X, labels, idx, eps, min_samples):
    neighbors = region_query(X, idx, eps)
    if len(neighbors) >= min_samples:
        return 'core'
    if labels[idx] >= 0:
        return 'border'
    return 'noise'
```
</details>

## Key takeaways

- **Density-based clustering:** core points (≥ `min_samples` within `eps`) seed clusters that
  grow by reachability; the rest is noise.
- **Matches sklearn** and correctly flags outliers (verified) — no need to pre-specify $k$.
- **Arbitrary shapes:** unlike K-Means, DBSCAN finds non-convex clusters and labels noise.
- **Eps is critical:** too small → all noise, too large → one cluster (demo) — use the
  k-distance elbow.